# MindRL Challenge 2026 — SPICE Starter Notebook

This notebook provides a starting point for solving the [MindRL Challenge](https://mindrl.org/) using the **SPICE** (Sparse and Interpretable Cognitive Equations) framework.

**Pipeline overview:**
1. **Data Loading** — Convert the MindRL JSONL format to a `SpiceDataset`
2. **Architecture Design** — Define `SpiceConfig` (which modules exist and what they receive) and `SpiceModel` (the trial-by-trial forward loop)
3. **Training** — Fit the model with joint RNN–SINDy optimization
4. **Analysis** — Inspect discovered equations, evaluate model fit, and analyze coefficient distributions

In [ ]:
import sys
from pathlib import Path

# Add project root to path so weinhardt2026 imports work
ROOT = Path.cwd().resolve().parents[2] if 'eckstein2026' in str(Path.cwd()) else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import torch

from spice import SpiceEstimator, SpiceConfig, BaseModel, csv_to_dataset

# Analysis imports (used after training)
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from weinhardt2026.analysis.analysis_morphing import run_morphing, get_morphed_coefficients

## 1. Data Loading

Convert the MindRL JSONL format into a `SpiceDataset`. The converter extracts participant IDs, block IDs, choices, rewards, and reaction times, then `csv_to_dataset()` builds the tensor dataset with one-hot encoded actions and rewards.

In [ ]:
"""Convert MindRL JSONL dataset to SPICE-compatible CSV.

JSONL format (one trajectory per line):
    {
        "context": {"subject_id": "sub_000848", "metadata": {"block_id": "block_003816"}, ...},
        "trials": [{"trial_index": 0, "action": 1, "reward": 8.0, "info": {"rt": 440.0}}, ...]
    }

Output CSV format (matching eckstein2024.csv):
    participant, block, trial_id, choice, reward, rt, mean_reward, mean_rt
"""

import json
from pathlib import Path

import pandas as pd


def convert_mindrl_jsonl_to_csv(input_path: str, output_path: str = None) -> pd.DataFrame:
    rows = []

    with open(input_path) as f:
        for line in f:
            data = json.loads(line)
            ctx = data["context"]

            subject_id = int(ctx["subject_id"].replace("sub_", ""))
            block_id = int(ctx["metadata"]["block_id"].replace("block_", ""))

            for trial in data["trials"]:
                rows.append(
                    {
                        "participant": subject_id,
                        "block": block_id,
                        "trial_id": trial["trial_index"],
                        "choice": trial["action"],
                        "reward": trial["reward"] / 100.0,
                        "rt": trial["info"].get("rt"),
                    }
                )

    df = pd.DataFrame(rows)
    # Renumber blocks sequentially (1, 2, 3, ...) per participant
    df["block"] = df.groupby("participant")["block"].transform(
        lambda s: pd.factorize(s, sort=True)[0] + 1
    )
    # Per-participant summary statistics (used as criterion in individual differences analysis)
    df['mean_reward'] = df.groupby('participant')['reward'].transform('mean')
    df['mean_rt'] = df.groupby('participant')['rt'].transform('mean')

    if output_path is not None:
        df.to_csv(output_path, index=False)

    return df

In [ ]:
file_jsonl = 'data/public_train.jsonl'
file_csv = 'data/public_train.csv'
public_train_df = convert_mindrl_jsonl_to_csv(file_jsonl, file_csv)
dataset = csv_to_dataset(
    file=public_train_df,
    df_participant_id='participant',
    df_block='block',
    df_choice='choice',
    df_feedback='reward',
    additional_inputs=('mean_reward', 'mean_rt'),
)

## 2. SPICE Architecture Design

Design your cognitive model in two parts:

### 2.1 `SpiceConfig` — Declare modules, their SINDy library inputs, and memory states

`SpiceConfig` defines the *skeleton* of the cognitive model:
- **`library_setup`**: Maps each submodule name to the control signal names that enter its SINDy polynomial library. These names are labels — the actual tensor values are passed via `inputs` in `call_module()`.
- **`memory_state`**: The latent state variables the model maintains across trials (with initial values).
- **`states_in_logit`**: Which memory states contribute to the final action logits. Auxiliary buffers (e.g., lagged values) should be excluded.

The config below is a minimal starting point. Extend it with more modules and control signals as needed.

In [ ]:
spice_config = SpiceConfig(
    library_setup={
        # Module name → tuple of SINDy library input signal names
        # (the module's own state is added automatically)
        'value_chosen':   ('reward[t]',),     # value update for chosen action
        'value_unchosen': (),                  # value update for unchosen actions (forgetting/decay)
        'value_choice':   ('action[t]',),      # choice perseveration / switching
    },
    memory_state={
        'value': 0.0,          # action value estimates
        'choice_value': 0.0,   # choice persistence trace
    },
    states_in_logit=['value', 'choice_value'],
)

### 2.2 `SpiceModel` — Trial-by-trial forward loop

The `SpiceModel` defines *how* modules are called each trial: which inputs they receive, which memory state they update, and which items are masked.

**Key design principle:** Precompute any derived features in the `forward()` loop *before* passing them to `call_module()`. This makes the dynamics more polynomial-amenable — the RNN submodules should only need to learn simple update rules, not complex feature transformations.

**Feature computation examples** (from the Eckstein 2026 study model):

```python
# 1. Broadcast reward to all items (environment-level reward signal)
reward_full = spice_signals.feedback.sum(dim=-1, keepdim=True).expand_as(spice_signals.actions)

# 2. Mean value across items (reference point for relative comparisons)
mean_value = self.state['value'].mean(dim=-1, keepdim=True).expand_as(self.state['value']).detach()

# 3. Value change detection (split into positive and negative components)
dvalue = (self.state['value'] - self.state['value[t-1]']).detach()
dvalue_pos = torch.relu(dvalue)    # positive surprise
dvalue_neg = torch.relu(-dvalue)   # negative surprise

# 4. Spatial features (circular distance between chosen and each item)
chosen_idx = spice_signals.actions[trial].argmax(dim=-1, keepdim=True)
items = torch.arange(self.n_actions, device=self.device).expand_as(spice_signals.actions[trial])
raw_dist = torch.abs(items - chosen_idx)
circ_dist = torch.min(raw_dist, self.n_actions - raw_dist)
is_adjacent = (circ_dist == 1).float()
is_opposite = (circ_dist == (self.n_actions // 2)).float()

# 5. Temporal buffers (store previous state for lagged features)
self.state['value[t-1]'] = self.state['value']
self.state['action[t-1]'] = spice_signals.actions[trial]
```

Use `action_mask` to route updates to the correct items:
- `action_mask=spice_signals.actions[trial]` → updates only the **chosen** item
- `action_mask=1 - spice_signals.actions[trial]` → updates only **unchosen** items
- `action_mask=None` → updates **all** items

In [ ]:
class SpiceModel(BaseModel):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        
        # Default modules from SpiceConfig are initialized automatically during __init__.
        # Override individual modules here if you need custom settings, e.g.:
        # self.setup_module(key_module='value_chosen', input_size=1, dropout=0.1, polynomial_degree=2)
        
    def forward(self, inputs, prev_state=None):
        
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embeddings = self.participant_embedding(spice_signals.participant_ids)
        
        # --- Precompute any derived features here (before the trial loop) ---
        # Example: broadcast partial-feedback reward to all-item format
        # reward_full = spice_signals.feedback.sum(dim=-1, keepdim=True).expand_as(spice_signals.actions)
        
        for trial in spice_signals.trials:
            
            # --- Compute trial-specific features ---
            # Example: mean value as a reference point
            # mean_value = self.state['value'].mean(dim=-1, keepdim=True).expand_as(self.state['value']).detach()
            
            # --- Update chosen action values ---
            self.call_module(
                key_module='value_chosen',
                key_state='value',
                action_mask=spice_signals.actions[trial],       # only update chosen item
                inputs=(spice_signals.feedback[trial],),        # reward for chosen action
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # --- Update unchosen action values ---
            self.call_module(
                key_module='value_unchosen',
                key_state='value',
                action_mask=1 - spice_signals.actions[trial],   # only update unchosen items
                inputs=(),
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # --- Update choice persistence ---
            self.call_module(
                key_module='value_choice',
                key_state='choice_value',
                inputs=(spice_signals.actions[trial],),         # current choice as input
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embeddings,
            )
            
            # --- Compose logits from memory states ---
            spice_signals.logits[trial] = self.state['value'] + self.state['choice_value']
            
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

## 3. Training

Configure the `SpiceEstimator` and fit the model. Training proceeds in two stages:
1. **Joint RNN–SINDy training** — the RNN learns to predict behavior while SINDy regularization pushes dynamics toward sparse polynomial form. Coefficients are pruned periodically.
2. **Final SINDy refit** — RNN weights are frozen and SINDy coefficients are recalibrated on the learned hidden states.

### OOS-ready training

The challenge evaluates on **out-of-sample participants** (unseen IDs). By default, `csv_to_dataset` remaps participant IDs to a compact 0..N-1 range, so new IDs at test time have no embedding slot.

The cell below fixes this by:
1. Restoring original participant IDs in the dataset (so ID 848 stays 848, not some arbitrary index)
2. Setting `n_participants = max_id + 1` so the embedding table has a slot for every possible ID
3. After training, filling all **untrained** embedding slots (and SINDy coefficients) with the population mean — giving unseen participants a reasonable default

In [ ]:
# --- Restore original participant IDs in dataset ---
# csv_to_dataset remaps IDs to 0..N-1; we need the original IDs so that
# unseen participant IDs at test time map to a valid (mean-initialized) embedding slot.

import numpy as np

# Build reverse mapping: normalized_id -> original_id
original_ids = public_train_df['participant'].unique()  # encounter order matches _normalize_ids
normalized_to_original = {idx: orig_id for idx, orig_id in enumerate(original_ids)}

# Replace normalized IDs with original IDs in the dataset
for session_idx in range(dataset.xs.shape[0]):
    norm_id = int(dataset.xs[session_idx, 0, 0, -1].item())
    dataset.xs[session_idx, :, :, -1] = normalized_to_original[norm_id]

# Update n_participants to cover the full ID range (max_id + 1)
max_participant_id = int(dataset.xs[..., -1].max().item())
dataset.n_participants = max_participant_id + 1
trained_ids = sorted(set(int(dataset.xs[s, 0, 0, -1].item()) for s in range(dataset.xs.shape[0])))

print(f"Unique trained participants: {len(trained_ids)}")
print(f"Max participant ID: {max_participant_id}")
print(f"Embedding table size (n_participants): {dataset.n_participants}")

In [ ]:
path_spice = 'spice_mindrl.pkl'

spice_estimator = SpiceEstimator(
    # Model
    spice_class=SpiceModel,
    spice_config=spice_config,

    # Data properties
    n_actions=dataset.n_actions,
    n_participants=dataset.n_participants,

    # Training
    epochs=1000,
    warmup_steps=500,
    # learning_rate=1e-2,
    # ensemble_size=10,

    # Sparsification
    # sindy_weight=0.01,
    # sindy_alpha=1e-4,
    # sindy_library_polynomial_degree=2,
    sindy_threshold_pruning=0.01,
    sindy_ensemble_pruning=0.5,

    verbose=True,
)

In [ ]:
spice_estimator.fit(dataset.xs, dataset.ys)
spice_estimator.save_spice(path_spice)

In [ ]:
# --- Stage 2 SINDy refit for unseen participants ---
# Set untrained embeddings to the population mean, then run a proper SINDy
# refit (epochs=0 skips Stage 1, only runs Stage 2).

model = spice_estimator.model

all_ids = set(range(dataset.n_participants))
untrained_ids = sorted(all_ids - set(trained_ids))
print(f"Untrained participant slots: {len(untrained_ids)}")

# 1. Set untrained embeddings to the population mean
with torch.no_grad():
    trained_idx = torch.tensor(trained_ids, device=model.device)
    emb_weight = model.participant_embedding.weight  # (E, P, D)
    mean_emb = emb_weight[:, trained_idx, :].mean(dim=1, keepdim=True)  # (E, 1, D)
    for pid in untrained_ids:
        emb_weight[:, pid, :] = mean_emb.squeeze(1)

# no need for that. all handled in stage 2
# 2. Set sparsity pattern to majority-vote across trained participants
# with torch.no_grad():
#     for key_module in model.sindy_coefficients:
#         presence = model.sindy_coefficients_presence[key_module]
#         majority_presence = (presence[:, trained_idx, :, :].float().mean(dim=1, keepdim=True) > 0.5)
#         for pid in untrained_ids:
#             presence[:, pid, :, :] = majority_presence.squeeze(1)

# 3. Create OOS dataset: all sessions mapped to one template untrained ID
template_id = untrained_ids[0]
xs_oos = dataset.xs.clone()
xs_oos[..., -1] = template_id
ys_oos = dataset.ys.clone()

# 4. Run fit with epochs=0 to skip Stage 1, only run Stage 2 SINDy refit
oos_estimator = SpiceEstimator(
    spice_class=SpiceModel,
    spice_config=spice_config,
    n_actions=dataset.n_actions,
    n_participants=dataset.n_participants,
    epochs=0,
    ensemble_size=model.ensemble_size,
    sindy_threshold_pruning=0.01,
    sindy_ensemble_pruning=0.5,
    verbose=True,
)
oos_estimator.model = model  # reuse the trained model
oos_estimator.fit(xs_oos, ys_oos)

# 5. Copy fitted coefficients from template to all other untrained IDs
with torch.no_grad():
    for key_module in model.sindy_coefficients:
        template_coeffs = model.sindy_coefficients[key_module].data[:, template_id, :, :].clone()
        template_presence = model.sindy_coefficients_presence[key_module][:, template_id, :, :].clone()
        for pid in untrained_ids[1:]:
            model.sindy_coefficients[key_module].data[:, pid, :, :] = template_coeffs
            model.sindy_coefficients_presence[key_module][:, pid, :, :] = template_presence

spice_estimator.model = model
print("Saving OOS-ready model...")
spice_estimator.save_spice(path_spice)

In [ ]:
spice_estimator.load_spice(path_spice)

## 4. Analysis

### 4.1 Discovered Equations

Print the symbolic equations discovered by SPICE for a few example participants.

In [ ]:
spice_estimator.eval()

for pid in range(min(3, dataset.n_participants)):
    print(f"\n--- Participant {pid} ---")
    spice_estimator.print_spice_model(participant_id=pid)

### 4.2 Model Evaluation

Compare the SPICE model (both RNN and equation mode) against baselines using information-theoretic metrics (NLL, AIC, BIC). Pass additional benchmark models via `benchmark_model` or `gru_model` if available.

PS: As a general benchmark value you can orient yourself at my SpiceModel for this task which achieves roughly 60% averaged trial likelihood.

In [ ]:
results = analysis_model_evaluation(
    dataset=dataset,
    spice_model=spice_estimator,
    output_dir='results',
)
print(results)

### 4.3 Coefficient Distributions

Analyze the distribution of SINDy coefficients across ensemble members and participants. This produces:
- **Ensemble consistency** metrics (coefficient of variation, presence agreement)
- **Violin plots** of nonzero coefficient values
- **Sparsity heatmap** (participants x terms)

In [ ]:
analysis_coefficients_distributions(
    spice_model=spice_estimator,
    output_dir='results',
)

### 4.4 Individual Differences

Relate SINDy coefficient patterns to a continuous behavioral criterion (e.g., mean reward). This runs logistic regression on coefficient presence and non-parametric tests on coefficient magnitudes to identify which equation terms correlate with the criterion variable.

Adjust `criterion` to any numeric column in the original CSV, or use `analysis='disc'` with a categorical column for group comparisons.

In [ ]:
analysis_coefficients_individuals(
    spice_model=spice_estimator,
    path_data=file_csv,
    analysis='cont',
    criterion='mean_reward',
    output_dir='results',
)

## 5. Model Morphing

Model morphing reveals how the *structure* of discovered equations changes along a behavioral dimension (e.g., average reward). The pipeline:

1. Finds a direction in participant embedding space that best predicts a target metric (via linear regression)
2. Shifts each participant's embedding along that direction in discrete steps
3. Refits SINDy equations at each step using a fast ridge–prune cycle (no SGD)
4. Aggregates across ensemble members to produce mean ± SE coefficient curves

This shows which equation terms appear/disappear and how their magnitudes change as we move from low- to high-performing participants — revealing structural (not just parametric) individual differences.

In [ ]:
# Run morphing analysis across all ensemble members
morphing_result = run_morphing(
    estimator=spice_estimator,
    dataset=dataset,
    metric_values=dataset.xs[:, 0, 0, dataset.n_actions*2],  # gets the mean reward per participant from the dataset
    n_steps=20,
    morphing_range_sd=1.0,
    save_dir='params/morphing/',
    verbose=True,
)

In [ ]:
# Aggregate morphed coefficients across ensemble members
morph_coeffs = get_morphed_coefficients(morphing_result)

for module, data in morph_coeffs.items():
    n_active = (data['inclusion_probability'] > 0).sum(axis=1)
    print(f"{module}: {len(data['term_names'])} terms, "
          f"active per step: [{n_active.min():.0f}, {n_active.max():.0f}]")